# Numerical Stability

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 06.01 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/02_numerical_stability.ipynb)

---

## 🎯 Learning Objective

Learn catastrophic cancellation and the log-sum-exp trick, and derive a numerically stable softmax and cross-entropy implementation.

---

## 📐 Theory

`06.01` established that floats are finite approximations. This notebook covers what happens
when an otherwise-correct *formula* gets evaluated using those approximations in the wrong
order — and how to reorder the arithmetic so the same mathematical answer survives intact.

### Catastrophic cancellation

Subtracting two nearly-equal large numbers can destroy almost all of the result's meaningful
digits. Consider $(10^{20} + 1) - 10^{20}$: mathematically this is exactly $1$. But in floating
point, $10^{20} + 1$ first rounds to the nearest representable float32/float64 value — and
$1$ is *far* smaller than $10^{20}$'s machine epsilon-scaled gap between representable numbers
(recall `06.01`), so the addition doesn't move the stored value at all. Subtracting $10^{20}$
back out then returns $0$, not $1$ — the "+1" was lost the instant it was added, not during the
subtraction. **Catastrophic cancellation** is this general phenomenon: whenever a computation
subtracts two nearly-equal quantities, whatever rounding error was already present in each one
individually gets massively amplified in *relative* terms, because the (small) true difference
is now being measured against a (large) pre-existing rounding error.

### The log-sum-exp trick

A recurring pattern in probability and ML is computing $\log\sum_i e^{x_i}$ — the normalizing
constant behind softmax, log-likelihoods, and more. Evaluated naively, $e^{x_i}$ overflows to
`inf` the moment any $x_i$ exceeds about $709$ (where $e^{709}$ hits float64's max), even though
the final *logarithm* of that sum might be a perfectly ordinary, modest number. The fix uses
one algebraic identity:

$$\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i - m}, \qquad m = \max_i x_i$$

Subtracting the max *before* exponentiating guarantees every exponent argument is $\le 0$, so
every $e^{x_i-m} \in (0, 1]$ — no overflow is possible, and the identity is exact algebra, not
an approximation (you can verify it by factoring $e^m$ out of the sum and taking the log). This
is the **log-sum-exp trick**, and it's one of the most-used stability tricks in all of applied
probability and machine learning.

### Numerically stable softmax

Softmax, $\text{softmax}(x)_i = e^{x_i} / \sum_j e^{x_j}$, has the exact same overflow problem
as log-sum-exp, since it's built from the same $e^{x_i}$ terms. The fix is the identical
subtract-the-max trick:

$$\text{softmax}(x)_i = \frac{e^{x_i-m}}{\sum_j e^{x_j-m}}, \qquad m=\max_j x_j$$

This is provably the *same* function as the naive formula — multiplying every numerator and
denominator term by the constant $e^{-m}$ doesn't change the ratio — but it's numerically safe:
the largest exponent argument is now exactly $0$ (giving $e^0=1$, never overflowing), and every
other argument is $\le 0$.

### Numerically stable cross-entropy

Cross-entropy loss (recall `05.03`) combines a $\log$ with a softmax:
$-\log(\text{softmax}(x)_y)$ for the true class $y$. Computed as two separate steps (softmax,
then log), this risks a second failure mode: if softmax legitimately produces a very small
probability (a confident wrong prediction), taking its log can lose precision or even hit
$\log(0)=-\infty$. The fix is to never materialize the intermediate softmax probabilities at
all — combine the softmax and the log analytically into a single **log-softmax** formula built
directly from log-sum-exp:

$$\log\text{softmax}(x)_y = x_y - \log\sum_j e^{x_j} = x_y - \left(m + \log\sum_j e^{x_j-m}\right)$$

This is exactly what `torch.nn.functional.cross_entropy` and `log_softmax` compute internally —
never forming the (potentially tiny, precision-losing) intermediate probability at all.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting naive vs. stable softmax's output as the input magnitude grows shows the naive
version collapsing into NaN well before the stable version shows any sign of trouble.

In [ ]:
# As we scale up the SAME 3-logit pattern (only relative differences matter for softmax's
# true output), naive softmax breaks down long before stable softmax does.
def naive_softmax(x):
    e = np.exp(x)
    return e / e.sum()

def stable_softmax(x):
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

base_logits = np.array([1.0, 2.0, 3.0])  # relative shape stays fixed; only overall SCALE grows
scales = np.linspace(1, 300, 80)
naive_p0, stable_p0, naive_has_nan = [], [], []
for s in scales:
    logits = base_logits * s
    with np.errstate(over='ignore', invalid='ignore'):
        naive_result = naive_softmax(logits)
    naive_p0.append(naive_result[0])
    naive_has_nan.append(np.any(np.isnan(naive_result)))
    stable_p0.append(stable_softmax(logits)[0])
naive_has_nan = np.array(naive_has_nan)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scales, stable_p0, color="#4C72B0", lw=2.5, label="stable softmax (subtract max first)")
ax.plot(scales, naive_p0, color="#C44E52", lw=1.5, ls="--", label="naive softmax (exp then normalize)")
if naive_has_nan.any():
    ax.axvline(scales[naive_has_nan.argmax()], color="gray", ls=":", label="naive first produces NaN here")
ax.set_xlabel("scale factor applied to logits [1,2,3]"); ax.set_ylabel("softmax output, class 0")
ax.set_title("Naive softmax silently returns 0 (then NaN) as exp() overflows;\nstable softmax never does")
ax.legend()
plt.show()

if naive_has_nan.any():
    first_nan_scale = scales[naive_has_nan.argmax()]
    print(f"Naive softmax first produces a NaN entry around scale factor ~ {first_nan_scale:.1f}")
else:
    print("Naive softmax did not hit NaN in this range, but see the numbers below --")
print(f"even BEFORE that point, naive softmax already returns exactly 0.0 for class 0 well before")
print(f"scale={scales[naive_has_nan.argmax()] if naive_has_nan.any() else scales[-1]:.0f}, which is")
print(f"arguably worse than a visible NaN: a silently WRONG (but plausible-looking) probability.")
print(f"Stable softmax's output stays well-defined and correctly proportioned at every scale tested.")

## 🐍 Implementation from Scratch

We reproduce catastrophic cancellation with real numbers, implement naive and stable versions
of softmax and log-sum-exp side by side, and confirm the stable versions match `scipy.special.logsumexp`
and never produce NaN/inf where the naive versions do.

In [ ]:
from scipy.special import logsumexp as scipy_logsumexp

# --- Catastrophic cancellation: a "+1" that gets silently dropped ---
huge = 1e20
result = (huge + 1.0) - huge
print(f"(1e20 + 1) - 1e20 = {result}  (mathematically exact answer is 1.0)")
print(f"The '+1' was lost the instant it was added -- 1e20's rounding gap (recall 06.01) is far")
print(f"bigger than 1.0, so 1e20 + 1.0 rounds right back down to 1e20 before any subtraction happens.")

# --- Naive vs stable softmax ---
def naive_softmax(x):
    e = np.exp(x)
    return e / e.sum()

def stable_softmax(x):
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

x_normal = np.array([1.0, 2.0, 3.0])
x_large = np.array([1000.0, 1001.0, 1002.0])   # same RELATIVE differences, much larger scale

print(f"\nOn modest inputs {x_normal}:")
print(f"  naive:  {naive_softmax(x_normal)}")
print(f"  stable: {stable_softmax(x_normal)}")
print(f"  agree: {np.allclose(naive_softmax(x_normal), stable_softmax(x_normal))}")

with np.errstate(over='ignore', invalid='ignore'):
    naive_result = naive_softmax(x_large)
print(f"\nOn large inputs {x_large}:")
print(f"  naive:  {naive_result}  <- NaN, because exp(1000) already overflowed to inf")
print(f"  stable: {stable_softmax(x_large)}  <- still a valid probability distribution")

# --- Naive vs stable log-sum-exp, cross-checked against scipy's implementation ---
def stable_logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

with np.errstate(over='ignore'):
    naive_lse = np.log(np.sum(np.exp(x_large)))
stable_lse = stable_logsumexp(x_large)
scipy_lse = scipy_logsumexp(x_large)
print(f"\nlog-sum-exp on {x_large}:")
print(f"  naive:  {naive_lse}  <- infinite, exp() overflowed before the log could rescue it")
print(f"  stable: {stable_lse:.6f}")
print(f"  scipy:  {scipy_lse:.6f}")
print(f"  stable matches scipy: {np.isclose(stable_lse, scipy_lse)}")

## 🤖 Why This Matters for AI

Every production cross-entropy loss implementation — `torch.nn.functional.cross_entropy`,
`torch.nn.CrossEntropyLoss`, TensorFlow's `softmax_cross_entropy_with_logits` — uses the
log-sum-exp trick internally, computing a fused log-softmax rather than softmax followed by a
separate log. A model that produces a single unusually large logit (common early in training,
or with an unstable learning rate, recall `04.06`) would otherwise crash training outright with
a NaN loss, rather than simply reporting a large-but-finite one. Below, we implement a
numerically stable cross-entropy from scratch, and stress-test it against `torch.nn.functional.cross_entropy`
on logits large enough to make the naive two-step version fail outright.

In [ ]:
import torch

def stable_log_softmax(x):
    # This is the fused log-softmax mentioned in the Theory section: never materializes
    # an intermediate probability, so it can't underflow OR overflow independently.
    m = np.max(x, axis=-1, keepdims=True)
    return x - m - np.log(np.sum(np.exp(x - m), axis=-1, keepdims=True))

def stable_cross_entropy(logits, target_idx):
    log_probs = stable_log_softmax(logits)
    return -log_probs[np.arange(len(logits)), target_idx]

rng = np.random.default_rng(0)
# Deliberately huge logits -- exactly the kind of spike an unstable training run could produce
logits = rng.normal(scale=500, size=(4, 5))
targets = np.array([0, 2, 4, 1])

ours = stable_cross_entropy(logits, targets)
torch_ce = torch.nn.functional.cross_entropy(
    torch.tensor(logits), torch.tensor(targets), reduction='none'
).numpy()

print(f"Logits scale: {np.abs(logits).max():.1f} (would make naive exp() overflow instantly)")
print(f"\nOur from-scratch stable cross-entropy: {np.round(ours, 4)}")
print(f"torch.nn.functional.cross_entropy:      {np.round(torch_ce, 4)}")
print(f"Match: {np.allclose(ours, torch_ce, atol=1e-4)}")
print(f"Any NaN in our result: {np.any(np.isnan(ours))}")

# Contrast: the naive two-step version DOES fail on these same logits
with np.errstate(over='ignore', invalid='ignore'):
    naive_probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    naive_ce = -np.log(naive_probs[np.arange(len(logits)), targets])
print(f"\nNaive (unstable) cross-entropy on the SAME logits: {naive_ce}")
print("The naive version fails exactly where the stable one succeeds -- same math, different arithmetic.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, compute `stable_softmax` on $x=(5, 1, -3)$: subtract the max, exponentiate each
   shifted value, sum, and divide. Check your answer sums to exactly $1$ and that the largest
   input gets the largest probability.

<details>
<summary>💡 Solution</summary>

$m=5$, shifted $=(0,-4,-8)$, $e^{\text{shifted}} = (1, 0.018316, 0.000335)$, sum $=1.018651$, so
$\text{softmax} \approx (0.9817, 0.0180, 0.0003)$ — sums to $1.0000$ exactly (up to
floating-point roundoff), and class 0 (the largest input) gets the largest probability, as
required.

</details>

### 💭 UNDERSTAND
2. `naive_softmax` on $x=(1000,1001,1002)$ fails (returns NaN). Someone proposes fixing it with
   `np.clip(x, -50, 50)` before exponentiating instead of the max-subtraction trick. Predict what
   `clipped_softmax` returns on this input, then check. Why does clipping give a qualitatively
   *worse* wrong answer than simply failing with a NaN?

<details>
<summary>💡 Solution</summary>

Clipping maps all three logits to the *same* value ($50$), since all three exceed the clip
ceiling — so `clipped_softmax` returns exactly $(1/3, 1/3, 1/3)$, a uniform distribution that
throws away the true relative differences between the logits entirely. This is worse than a
NaN: a NaN is loud and obviously broken, while a silently-uniform softmax looks like a
completely plausible, valid probability distribution — it just happens to be wrong, and nothing
in the output signals that. Max-subtraction preserves the *relative* gaps between logits exactly
(it only removes the shared additive constant, which cancels in the softmax ratio); clipping
destroys any relative gap larger than the clip window.

</details>

### ✏️ DERIVE
3. Prove that $\frac{\partial}{\partial x_i} \log\sum_j e^{x_j} = \text{softmax}(x)_i$. Then
   verify your result numerically: compare a central finite-difference estimate of
   $\nabla_x \, \text{logsumexp}(x)$ against `stable_softmax(x)` directly.

<details>
<summary>💡 Solution outline</summary>

Let $L(x) = \log\sum_j e^{x_j}$. By the chain rule,
$\frac{\partial L}{\partial x_i} = \frac{1}{\sum_j e^{x_j}} \cdot \frac{\partial}{\partial x_i}\sum_j e^{x_j} = \frac{e^{x_i}}{\sum_j e^{x_j}} = \text{softmax}(x)_i$
— the log-sum-exp function's gradient *is* softmax, which is exactly why softmax shows up as the
backward pass of a log-sum-exp-based loss. Numerically, on $x=(3,5,1)$, a central finite
difference with $\epsilon=10^{-6}$ gives $\approx (0.11731,\ 0.86681,\ 0.01588)$, matching
`stable_softmax(x)` to within $10^{-6}$.

</details>

### 🐍 IMPLEMENT
4. Implement a numerically stable **sigmoid**, $\sigma(x) = 1/(1+e^{-x})$, that avoids overflow
   for large-magnitude $x$ of either sign, using a case split (a different, algebraically
   equivalent formula for $x \ge 0$ vs. $x < 0$, so the argument of every `exp` call is always
   $\le 0$).

<details>
<summary>✅ How to know you're right</summary>

Your result should match `scipy.special.expit` (to `atol=1e-12`) across a sweep including
$x=-1000$ and $x=1000$. As a sharper check, run the *naive* one-line formula under
`np.errstate(over='raise')`: it raises `FloatingPointError: overflow encountered in exp` at
$x=-1000$, while your stable version raises nothing anywhere in the sweep — that's the actual
bug being fixed, not just a cosmetic warning.

</details>

### 🤖 APPLY
5. A 2-component Gaussian mixture model with equal weights $0.5/0.5$ and both components
   $\mathcal{N}(0, 1)$ is asked for its log-likelihood at $x=100$ — deep in both components'
   tails. Compute this via (a) the naive two-step way (sum the densities, then take one log) and
   (b) `logsumexp` on the log-densities directly. Which one is usable, and why does the modest
   input $x=100$ (not an extreme value like $x=10^6$) already break the naive approach?

<details>
<summary>✅ What you should observe</summary>

The naive approach computes each component's density as $\exp(-5000.92)$, which underflows to
exactly $0.0$ in float64 (since $e^{-5000.92}$ is astronomically below the smallest representable
positive float64), so the summed density is $0.0$ and its log is $-\infty$ — a real number
turned into "no signal at all." The `logsumexp` route never exponentiates the raw $-5000.92$; it
stays in log-space throughout and returns the correct, finite value $\approx -5000.919$. The
lesson: you don't need astronomically extreme inputs to break naive exponentiation — a log-density
of a few thousand (utterly ordinary for a moderately-out-of-distribution point under a sharp
distribution) is already enough.

</details>

### 🚀 CHALLENGE
6. The naive one-pass variance formula $\text{Var}(X) \approx E[X^2] - E[X]^2$ is a textbook
   example of catastrophic cancellation. Using `float32`, generate $n=10000$ samples from
   $\mathcal{N}(\mu, 2^2)$ for $\mu \in \{10^5, 10^6, 10^8\}$ and compare the naive formula,
   the two-pass (mean-centered) formula, and the true variance ($\approx 4$, computed in
   float64). At which $\mu$ does the naive formula return a *negative* variance — a value that's
   not just imprecise but literally impossible? Does switching to the two-pass formula fully fix
   the problem at every $\mu$ you tried, or does something else start to go wrong at the largest
   scale, for a different reason?

<details>
<summary>🔍 What a strong answer covers</summary>

The naive formula returns an impossible negative variance at $\mu=10^5$ ($\approx -1024$) and
at $\mu=10^8$ ($\approx -1.07\times 10^9$) — both catastrophic cancellation, since $E[X^2]$ and
$E[X]^2$ are both close to $\mu^2$ and their float32-rounded difference is dominated by rounding
error rather than the true $O(\sigma^2)$ gap. The two-pass formula fixes this at $\mu=10^5$
(recovers $\approx 3.98$, matching the true variance), but at $\mu=10^8$ it *also* degrades
(returns $\approx 67$, not $4$) — for a genuinely different reason: `float32`'s representable-value
gap near $10^8$ is $8.0$ (via `np.spacing(np.float32(1e8))`), which is *larger* than the true
standard deviation of $2.0$. At that point individual data points can no longer be distinguished
from their neighbors within $\pm 4\sigma$ once stored in float32 — no amount of rearranging the
arithmetic can recover information the storage format itself already destroyed. A strong answer
identifies that this second failure is a *representation* problem, not an *algorithm* problem,
and that fixing it requires higher precision storage (float64) or centering the data before
casting to float32, not a cleverer variance formula.

</details>

---

## 📚 Further Reading
- Goodfellow, Bengio & Courville, *Deep Learning*, Ch. 4.1 (Overflow and Underflow)
- Blanchard, Higham & Higham, ["Accurately Computing the Log-Sum-Exp and Softmax Functions"](https://arxiv.org/abs/1909.03469)

---

*Next notebook: [Numerical Differentiation](03_numerical_differentiation.ipynb)*